# Telecom Network Fault Prediction & Machine Learning Pipeline

In [1]:

import os
import pandas as pd

alarms = pd.read_excel('alarm.xlsx')
complaints = pd.read_excel('complain.xlsx')
chains = pd.read_excel('chain.xlsx')
hierarchy = pd.read_excel('network.xlsx')
weather = pd.read_excel('weather.xlsx')

In [2]:

datasets = [alarms, complaints, chains, hierarchy, weather]

for df in datasets:
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
    )

In [3]:

alarms['alarm_timestamp'] = pd.to_datetime(alarms['alarm_timestamp'])
complaints['complaint_time'] = pd.to_datetime(complaints['complaint_time'])
weather['event_timestamp'] = pd.to_datetime(weather['event_timestamp'])

In [4]:

master = alarms.merge(
    hierarchy,
    on='network_node',
    how='left',
    suffixes=('', '_hierarchy')
)
master = master.merge(
    weather,
    on='network_node',
    how='left',
    suffixes=('', '_weather')
)
master = master.merge(
    complaints,
    left_on='alarm_id',
    right_on='related_alarm_id',
    how='left'
)
master = master.merge(
    chains,
    left_on='alarm_id',
    right_on='parent_alarm_id',
    how='left'
)

In [5]:

master.drop_duplicates(inplace=True)

In [6]:

master.fillna(
    {
        'complaint_type': 'No Complaint',
        'weather_condition': 'Unknown',
        'dependency_strength': 0,
        'root_cause_probability': 0
    },
    inplace=True
)

In [7]:

master['hour'] = master['alarm_timestamp'].dt.hour
master['day'] = master['alarm_timestamp'].dt.day
master['month'] = master['alarm_timestamp'].dt.month
master['weekday'] = master['alarm_timestamp'].dt.day_name()
master['is_weekend'] = (
    master['alarm_timestamp'].dt.dayofweek >= 5
).astype(int)

In [8]:

os.makedirs('data', exist_ok=True)
master.to_csv('master_dataset.csv', index=False)
master.to_csv('data/master_dataset.csv', index=False)
print('Master dataset shape:', master.shape)

Master dataset shape: (56458, 81)


## Exploratory Data Analysis (EDA)

In [9]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 6)

master = pd.read_csv('master_dataset.csv')
master.head()

,alarm_id,alarm_timestamp,alarm_type,severity,technology_x,vendor,region_x,circle_x,city_x,hub_id,...,child_alarm_type,network_node,propagation_delay_minutes,dependency_strength,root_cause_probability,hour,day,month,weekday,is_weekend
0,ALM000001,2025-01-13 20:45:00,Synchronization Failure,Major,4G LTE,Nokia,North,Delhi,Delhi,HUB163,...,NaN,NaN,NaN,0.0,0.00,20,13,1,Monday,0
1,ALM000002,2025-07-28 08:44:00,Battery Low,Critical,4G LTE,Nokia,South,Karnataka,Bengaluru,HUB052,...,NaN,NaN,NaN,0.0,0.00,8,28,7,Monday,0
2,ALM000003,2025-03-26 23:34:00,Packet Loss,Minor,FTTH,Ericsson,North,Haryana,Gurugram,HUB073,...,Link Down,NODE46795,28.0,0.8,0.91,23,26,3,Wednesday,0
3,ALM000003,2025-03-26 23:34:00,Packet Loss,Minor,FTTH,Ericsson,North,Haryana,Gurugram,HUB073,...,Link Down,NODE46795,28.0,0.8,0.91,23,26,3,Wednesday,0
4,ALM000004,2024-09-13 11:59:00,High Temperature,Minor,4G LTE,Ericsson,West,Maharashtra,Mumbai,HUB097,...,NaN,NaN,NaN,0.0,0.00,11,13,9,Friday,0


In [10]:
print('Rows :', master.shape[0])
print('Columns :', master.shape[1])

Rows : 56458
Columns : 81


In [11]:
master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56458 entries, 0 to 56457
Data columns (total 81 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   alarm_id                    56458 non-null  object 
 1   alarm_timestamp             56458 non-null  object 
 2   alarm_type                  56458 non-null  object 
 3   severity                    56458 non-null  object 
 4   technology_x                56458 non-null  object 
 5   vendor                      56458 non-null  object 
 6   region_x                    56458 non-null  object 
 7   circle_x                    56458 non-null  object 
 8   city_x                      56458 non-null  object 
 9   hub_id                      56458 non-null  object 
 10  router_id                   56458 non-null  object 
 11  bts_id                      56458 non-null  object 
 12  cell_id                     56458 non-null  object 
 13  network_node_x              564

In [12]:
master.describe(include="all")

,alarm_id,alarm_timestamp,alarm_type,severity,technology_x,vendor,region_x,circle_x,city_x,hub_id,...,child_alarm_type,network_node,propagation_delay_minutes,dependency_strength,root_cause_probability,hour,day,month,weekday,is_weekend
count,56458,56458,56458,56458,56458,56458,56458,56458,56458,56458,...,11095,11095,11095.000000,56458.000000,56458.000000,56458.000000,56458.000000,56458.000000,56458,56458.000000
unique,50000,48795,12,4,4,3,3,4,4,200,...,10,9071,NaN,NaN,NaN,NaN,NaN,NaN,7,NaN
top,ALM032152,2024-02-22 04:13:00,Packet Loss,Major,FTTH,Huawei,North,Haryana,Gurugram,HUB106,...,Fiber Cut,NODE29796,NaN,NaN,NaN,NaN,NaN,NaN,Tuesday,NaN
freq,12,12,4811,14159,14299,18909,28339,14194,14194,330,...,1131,6,NaN,NaN,NaN,NaN,NaN,NaN,8117,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,15.323028,0.147325,0.146322,11.465798,15.627263,6.220164,NaN,0.285965
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,8.704482,0.304845,0.302505,6.905595,8.751495,3.302427,NaN,0.451877
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,NaN,0.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,8.000000,0.000000,0.000000,5.000000,8.000000,3.000000,NaN,0.000000
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,15.000000,0.000000,0.000000,11.000000,16.000000,6.000000,NaN,0.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,23.000000,0.000000,0.000000,17.000000,23.000000,9.000000,NaN,1.000000


In [13]:
master.isnull().sum().sort_values(ascending=False)

region_weather          50986
humidity_percent        50986
weather_event_id        50986
circle_weather          50986
city_weather            50986
                        ...  
maintenance_required        0
weather_condition           0
power_status                0
previous_alarm              0
is_weekend                  0
Length: 81, dtype: int64

In [14]:
plt.figure(figsize=(12,6))
sns.heatmap(master.isnull(), cbar=False)
plt.title('Missing Values Heatmap')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/1126130136.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
print("Duplicate rows:", master.duplicated().sum())

Duplicate rows: 0


In [16]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=master,
    x='severity',
    order=master['severity'].value_counts().index
)

plt.title('Alarm Severity Distribution')
plt.xlabel('Severity')
plt.ylabel('Count')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/3828013312.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
plt.figure(figsize=(12,6))

sns.countplot(
    data=master,
    y='alarm_type',
    order=master['alarm_type'].value_counts().index
)

plt.title('Alarm Type Distribution')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/511764667.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
print("Columns in Master Dataset:", master.columns.tolist())

Columns in Master Dataset: ['alarm_id', 'alarm_timestamp', 'alarm_type', 'severity', 'technology_x', 'vendor', 'region_x', 'circle_x', 'city_x', 'hub_id', 'router_id', 'bts_id', 'cell_id', 'network_node_x', 'latitude', 'longitude', 'alarm_status', 'root_cause', 'duration_minutes', 'affected_customers', 'resolution_time', 'maintenance_required', 'weather_condition', 'power_status', 'previous_alarm', 'next_alarm', 'region_hierarchy', 'circle_hierarchy', 'city_hierarchy', 'hub_id_hierarchy', 'router_id_hierarchy', 'olt_id', 'bts_id_hierarchy', 'cell_id_hierarchy', 'vendor_hierarchy', 'technology_hierarchy', 'latitude_hierarchy', 'longitude_hierarchy', 'weather_event_id', 'event_timestamp', 'region_weather', 'circle_weather', 'city_weather', 'weather_condition_weather', 'temperature_c', 'humidity_percent', 'wind_speed_kmph', 'rainfall_mm', 'visibility_km', 'weather_severity', 'affected_alarm_probability', 'complaint_id', 'customer_id', 'complaint_time', 'network_node_y', 'related_alarm_id'

In [19]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=master,
    x='technology_x',
    order=master['technology_x'].value_counts().index
)

plt.title('technology_x-wise Alarms')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/1582011903.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=master,
    x='vendor'
)

plt.title('Vendor-wise Alarm Distribution')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/3493412893.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
plt.figure(figsize=(10,5))

sns.countplot(
    data=master,
    x='region_x',
    order=master['region_x'].value_counts().index
)

plt.title('region_x-wise Alarms')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/557085638.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
plt.figure(figsize=(12,6))

master['city_x'].value_counts().head(10).plot(kind='bar')

plt.title('Top 10 Cities with Maximum Alarms')
plt.ylabel('Number of Alarms')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/3049725887.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
plt.figure(figsize=(6,5))

master['alarm_status'].value_counts().plot(
    kind='pie',
    autopct='%1.1f%%'
)

plt.ylabel('')
plt.title('Alarm Status')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/186137444.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
plt.figure(figsize=(10,5))

sns.countplot(
    data=master,
    x='weather_condition',
    order=master['weather_condition'].value_counts().index
)

plt.xticks(rotation=30)
plt.title('Weather Conditions During Alarms')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/1531711142.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [25]:
plt.figure(figsize=(12,6))

sns.countplot(
    data=master,
    y='complaint_type',
    order=master['complaint_type'].value_counts().index
)

plt.title('Complaint Types')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/2159562689.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [26]:
plt.figure(figsize=(10,6))

sns.histplot(
    master['affected_customers'],
    bins=30,
    kde=True
)

plt.title('Affected Customers')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/1840439385.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [27]:
num_df = master.select_dtypes(include=np.number)
plt.figure(figsize=(12,8))

sns.heatmap(
    num_df.corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Matrix')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/1138823913.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [28]:
master['alarm_timestamp'] = pd.to_datetime(master['alarm_timestamp'])

daily = master.groupby(
    master['alarm_timestamp'].dt.date
).size()

plt.figure(figsize=(15,5))

daily.plot()

plt.title('Daily Alarm Trend')
plt.xlabel('Date')
plt.ylabel('Number of Alarms')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/1242920418.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [29]:
plt.figure(figsize=(10,6))

sns.countplot(
    data=master,
    x='technology_x',
    hue='severity'
)

plt.title('Severity by Technology')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/2924224655.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [30]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=master,
    x='vendor',
    hue='severity'
)

plt.title('Vendor vs Severity')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/2729306722.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [31]:
plt.figure(figsize=(12,6))

master['network_node'].value_counts().head(15).plot(kind='bar')

plt.title('Top Network Nodes with Most Alarms')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/1824389712.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### EDA Summary

• Total alarms analyzed: 56,458
• Most common alarm type: Packet Loss / Power Failure
• Highest severity level: CRITICAL / WARNING
• Top impacted regions & vendors: HUAWEI, FTTH Technology
• Most common weather conditions associated with alarms: Rain / Fog

## Model 1: Root Cause Classification Model

In [32]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

In [33]:
master = pd.read_csv('master_dataset.csv')

drop_cols_m1 = [
    'alarm_id',
    'alarm_timestamp',
    'previous_alarm',
    'parent_alarm_id',
    'child_alarm_id',
    'related_alarm_id',
    'event_timestamp',
    'complaint_time'
]

master_m1 = master.drop(columns=drop_cols_m1, errors='ignore')

target_m1 = 'root_cause'

X_m1 = master_m1.drop(columns=[target_m1])
y_m1 = master_m1[target_m1]

encoder_m1 = LabelEncoder()
y_m1_encoded = encoder_m1.fit_transform(y_m1)
print("Model 1 Target Classes:", list(encoder_m1.classes_))

Model 1 Target Classes: ['Battery Failure', 'Equipment Fault', 'Fiber Cut', 'Power Failure', 'Weather Impact']


In [34]:
cat_cols_m1 = X_m1.select_dtypes(include=['object', 'string']).columns
num_cols_m1 = X_m1.select_dtypes(exclude=['object', 'string']).columns

print(f"Categorical Columns ({len(cat_cols_m1)}):", list(cat_cols_m1))
print(f"Numerical Columns ({len(num_cols_m1)}):", list(num_cols_m1))

Categorical Columns (48): ['alarm_type', 'severity', 'technology_x', 'vendor', 'region_x', 'circle_x', 'city_x', 'hub_id', 'router_id', 'bts_id', 'cell_id', 'network_node_x', 'alarm_status', 'maintenance_required', 'weather_condition', 'power_status', 'next_alarm', 'region_hierarchy', 'circle_hierarchy', 'city_hierarchy', 'hub_id_hierarchy', 'router_id_hierarchy', 'olt_id', 'bts_id_hierarchy', 'cell_id_hierarchy', 'vendor_hierarchy', 'technology_hierarchy', 'weather_event_id', 'region_weather', 'circle_weather', 'city_weather', 'weather_condition_weather', 'weather_severity', 'complaint_id', 'customer_id', 'network_node_y', 'complaint_type', 'technology_y', 'customer_priority', 'status', 'region_y', 'circle_y', 'city_y', 'chain_id', 'parent_alarm_type', 'child_alarm_type', 'network_node', 'weekday']
Numerical Columns (24): ['latitude', 'longitude', 'duration_minutes', 'affected_customers', 'resolution_time', 'latitude_hierarchy', 'longitude_hierarchy', 'temperature_c', 'humidity_percen

In [35]:
numeric_transformer_m1 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

categorical_transformer_m1 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_m1 = ColumnTransformer(transformers=[
    ('num', numeric_transformer_m1, num_cols_m1),
    ('cat', categorical_transformer_m1, cat_cols_m1)
])

In [36]:
X_train_m1, X_test_m1, y_train_m1, y_test_m1 = train_test_split(
    X_m1,
    y_m1_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_m1_encoded
)

print('Training features shape:', X_train_m1.shape)
print('Testing features shape:', X_test_m1.shape)

Training features shape: (45166, 72)
Testing features shape: (11292, 72)


In [37]:
pipeline_m1 = Pipeline([
    ('preprocessor', preprocessor_m1),
    ('model', RandomForestClassifier(n_estimators=50, max_depth=20, random_state=42, n_jobs=-1))
])

In [38]:
print('Training Model 1 (Root Cause Classification)...')
pipeline_m1.fit(X_train_m1, y_train_m1)
print('Model 1 trained successfully!')

Training Model 1 (Root Cause Classification)...


Model 1 trained successfully!


In [39]:
y_pred_m1 = pipeline_m1.predict(X_test_m1)

acc_m1 = accuracy_score(y_test_m1, y_pred_m1)
print(f"Model 1 Accuracy: {acc_m1 * 100:.2f}%")
print()

print("Classification Report (Model 1 - Root Cause):")
target_names_m1 = [str(cls) for cls in encoder_m1.classes_]
print(classification_report(y_test_m1, y_pred_m1, target_names=target_names_m1))

Model 1 Accuracy: 22.22%

Classification Report (Model 1 - Root Cause):
                 precision    recall  f1-score   support

Battery Failure       0.35      0.04      0.08      2248
Equipment Fault       0.26      0.06      0.10      2267
      Fiber Cut       0.26      0.09      0.13      2252
  Power Failure       0.35      0.06      0.10      2239
 Weather Impact       0.21      0.85      0.33      2286

       accuracy                           0.22     11292
      macro avg       0.29      0.22      0.15     11292
   weighted avg       0.29      0.22      0.15     11292



In [40]:
import matplotlib.pyplot as plt
import seaborn as sns

cm_m1 = confusion_matrix(y_test_m1, y_pred_m1)
plt.figure(figsize=(10, 7))
sns.heatmap(
    cm_m1,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=encoder_m1.classes_,
    yticklabels=encoder_m1.classes_
)
plt.xlabel('Predicted Root Cause')
plt.ylabel('Actual Root Cause')
plt.title('Confusion Matrix - Model 1 (Root Cause Classification)')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/2616396668.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [41]:
cat_encoder_m1 = pipeline_m1.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
onehot_cols_m1 = list(cat_encoder_m1.get_feature_names_out(cat_cols_m1))
all_feature_names_m1 = list(num_cols_m1) + onehot_cols_m1

importances_m1 = pipeline_m1.named_steps['model'].feature_importances_
feat_imp_m1 = pd.Series(importances_m1, index=all_feature_names_m1).sort_values(ascending=False)

plt.figure(figsize=(12,6))
feat_imp_m1.head(15).plot(kind='barh').invert_yaxis()
plt.title('Top 15 Important Features for Model 1 (Root Cause)')
plt.xlabel('Importance Score')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/433322554.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Model 2: Future Alarm Prediction Model

In [42]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

In [43]:
master = pd.read_csv('master_dataset.csv')

target_m2 = 'next_alarm'

drop_cols_m2 = [
    'alarm_id',
    'alarm_timestamp',
    'previous_alarm',
    'parent_alarm_id',
    'child_alarm_id',
    'related_alarm_id',
    'event_timestamp',
    'complaint_time',
    target_m2
]

X_m2 = master.drop(columns=drop_cols_m2, errors='ignore')
y_m2 = master[target_m2]

encoder_m2 = LabelEncoder()
y_m2_encoded = encoder_m2.fit_transform(y_m2)
print("Model 2 Target Classes:", list(encoder_m2.classes_))

Model 2 Target Classes: ['BTS Down', 'Battery Low', 'Cell Down', 'DG Failure', 'Fiber Cut', 'High Temperature', 'Link Down', 'Packet Loss', 'Power Failure', 'Router Down', 'Synchronization Failure', 'Transmission Failure']


In [44]:
cat_cols_m2 = X_m2.select_dtypes(include=['object', 'string']).columns
num_cols_m2 = X_m2.select_dtypes(exclude=['object', 'string']).columns

print(f"Categorical Columns ({len(cat_cols_m2)}):", list(cat_cols_m2))
print(f"Numerical Columns ({len(num_cols_m2)}):", list(num_cols_m2))

Categorical Columns (48): ['alarm_type', 'severity', 'technology_x', 'vendor', 'region_x', 'circle_x', 'city_x', 'hub_id', 'router_id', 'bts_id', 'cell_id', 'network_node_x', 'alarm_status', 'root_cause', 'maintenance_required', 'weather_condition', 'power_status', 'region_hierarchy', 'circle_hierarchy', 'city_hierarchy', 'hub_id_hierarchy', 'router_id_hierarchy', 'olt_id', 'bts_id_hierarchy', 'cell_id_hierarchy', 'vendor_hierarchy', 'technology_hierarchy', 'weather_event_id', 'region_weather', 'circle_weather', 'city_weather', 'weather_condition_weather', 'weather_severity', 'complaint_id', 'customer_id', 'network_node_y', 'complaint_type', 'technology_y', 'customer_priority', 'status', 'region_y', 'circle_y', 'city_y', 'chain_id', 'parent_alarm_type', 'child_alarm_type', 'network_node', 'weekday']
Numerical Columns (24): ['latitude', 'longitude', 'duration_minutes', 'affected_customers', 'resolution_time', 'latitude_hierarchy', 'longitude_hierarchy', 'temperature_c', 'humidity_percen

In [45]:
numeric_transformer_m2 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

categorical_transformer_m2 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_m2 = ColumnTransformer(transformers=[
    ('num', numeric_transformer_m2, num_cols_m2),
    ('cat', categorical_transformer_m2, cat_cols_m2)
])

In [46]:
X_train_m2, X_test_m2, y_train_m2, y_test_m2 = train_test_split(
    X_m2,
    y_m2_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_m2_encoded
)

print('Training features shape:', X_train_m2.shape)
print('Testing features shape:', X_test_m2.shape)

Training features shape: (45166, 72)
Testing features shape: (11292, 72)


In [47]:
pipeline_m2 = Pipeline([
    ('preprocessor', preprocessor_m2),
    ('model', RandomForestClassifier(n_estimators=50, max_depth=20, random_state=42, n_jobs=-1))
])

In [48]:
print('Training Model 2 (Future Alarm Prediction)...')
pipeline_m2.fit(X_train_m2, y_train_m2)
print('Model 2 trained successfully!')

Training Model 2 (Future Alarm Prediction)...


Model 2 trained successfully!


In [49]:
y_pred_m2 = pipeline_m2.predict(X_test_m2)

acc_m2 = accuracy_score(y_test_m2, y_pred_m2)
print(f"Model 2 Accuracy: {acc_m2 * 100:.2f}%")
print()

print("Classification Report (Model 2 - Future Alarm Prediction):")
target_names_m2 = [str(cls) for cls in encoder_m2.classes_]
print(classification_report(y_test_m2, y_pred_m2, target_names=target_names_m2))

Model 2 Accuracy: 10.00%

Classification Report (Model 2 - Future Alarm Prediction):
                         precision    recall  f1-score   support

               BTS Down       0.50      0.03      0.05       926
            Battery Low       0.41      0.02      0.04       935
              Cell Down       0.31      0.02      0.03       939
             DG Failure       0.31      0.02      0.04       930
              Fiber Cut       0.13      0.04      0.06       960
       High Temperature       0.34      0.02      0.03       938
              Link Down       0.18      0.03      0.06       948
            Packet Loss       0.33      0.02      0.04       934
          Power Failure       0.30      0.03      0.05       952
            Router Down       0.09      0.92      0.16       959
Synchronization Failure       0.22      0.02      0.04       939
   Transmission Failure       0.26      0.02      0.03       932

               accuracy                           0.10     11292
   

In [50]:
import matplotlib.pyplot as plt
import seaborn as sns

cm_m2 = confusion_matrix(y_test_m2, y_pred_m2)
plt.figure(figsize=(12, 8))
sns.heatmap(
    cm_m2,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=encoder_m2.classes_,
    yticklabels=encoder_m2.classes_
)
plt.xlabel('Predicted Next Alarm')
plt.ylabel('Actual Next Alarm')
plt.title('Confusion Matrix - Model 2 (Future Alarm Prediction)')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/346592325.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [51]:
cat_encoder_m2 = pipeline_m2.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
onehot_cols_m2 = list(cat_encoder_m2.get_feature_names_out(cat_cols_m2))
all_feature_names_m2 = list(num_cols_m2) + onehot_cols_m2

importances_m2 = pipeline_m2.named_steps['model'].feature_importances_
feat_imp_m2 = pd.Series(importances_m2, index=all_feature_names_m2).sort_values(ascending=False)

plt.figure(figsize=(12,6))
feat_imp_m2.head(15).plot(kind='barh').invert_yaxis()
plt.title('Top 15 Important Features for Model 2 (Future Alarm Prediction)')
plt.xlabel('Importance Score')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/2094194081.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Model 3: Customer Impact Prediction Model (Regression)

In [52]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [53]:
master = pd.read_csv('master_dataset.csv')

target_m3 = 'affected_customers'

drop_cols_m3 = [
    'alarm_id',
    'alarm_timestamp',
    'previous_alarm',
    'parent_alarm_id',
    'child_alarm_id',
    'related_alarm_id',
    'event_timestamp',
    'complaint_time',
    target_m3
]

X_m3 = master.drop(columns=drop_cols_m3, errors='ignore')
y_m3 = master[target_m3]

print(f"Target column '{target_m3}' summary:")
print(y_m3.describe())

Target column 'affected_customers' summary:
count    56458.000000
mean      2500.615148
std       1438.593875
min         10.000000
25%       1257.000000
50%       2504.000000
75%       3734.000000
max       5000.000000
Name: affected_customers, dtype: float64


In [54]:
cat_cols_m3 = X_m3.select_dtypes(include=['object', 'string']).columns
num_cols_m3 = X_m3.select_dtypes(exclude=['object', 'string']).columns

numeric_transformer_m3 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

categorical_transformer_m3 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_m3 = ColumnTransformer(transformers=[
    ('num', numeric_transformer_m3, num_cols_m3),
    ('cat', categorical_transformer_m3, cat_cols_m3)
])

In [55]:
X_train_m3, X_test_m3, y_train_m3, y_test_m3 = train_test_split(
    X_m3,
    y_m3,
    test_size=0.2,
    random_state=42
)

print('Training features shape:', X_train_m3.shape)
print('Testing features shape:', X_test_m3.shape)

Training features shape: (45166, 72)
Testing features shape: (11292, 72)


In [56]:
pipeline_m3 = Pipeline([
    ('preprocessor', preprocessor_m3),
    ('model', RandomForestRegressor(n_estimators=30, max_depth=15, random_state=42, n_jobs=-1))
])

In [57]:
print('Training Model 3 (Customer Impact Prediction)...')
pipeline_m3.fit(X_train_m3, y_train_m3)
print('Model 3 trained successfully!')

Training Model 3 (Customer Impact Prediction)...


Model 3 trained successfully!


In [58]:
y_pred_m3 = pipeline_m3.predict(X_test_m3)

mae_m3 = mean_absolute_error(y_test_m3, y_pred_m3)
mse_m3 = mean_squared_error(y_test_m3, y_pred_m3)
rmse_m3 = np.sqrt(mse_m3)
r2_m3 = r2_score(y_test_m3, y_pred_m3)

print("=== Model 3 Regression Metrics ===")
print(f"Mean Absolute Error (MAE)  : {mae_m3:.2f}")
print(f"Mean Squared Error (MSE)   : {mse_m3:.2f}")
print(f"Root Mean Squared Error    : {rmse_m3:.2f}")
print(f"R² Score                   : {r2_m3:.4f}")

=== Model 3 Regression Metrics ===
Mean Absolute Error (MAE)  : 1239.71
Mean Squared Error (MSE)   : 2057956.19
Root Mean Squared Error    : 1434.56
R² Score                   : 0.0054


In [59]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
plt.scatter(y_test_m3, y_pred_m3, alpha=0.3, color='purple')
plt.plot([y_test_m3.min(), y_test_m3.max()], [y_test_m3.min(), y_test_m3.max()], 'r--', lw=2)
plt.xlabel('Actual Affected Customers')
plt.ylabel('Predicted Affected Customers')
plt.title('Actual vs Predicted Customer Impact (Model 3)')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/2223310020.py:4: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(10, 6))
/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/2223310020.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [60]:
cat_encoder_m3 = pipeline_m3.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
onehot_cols_m3 = list(cat_encoder_m3.get_feature_names_out(cat_cols_m3))
all_feature_names_m3 = list(num_cols_m3) + onehot_cols_m3

importances_m3 = pipeline_m3.named_steps['model'].feature_importances_
feat_imp_m3 = pd.Series(importances_m3, index=all_feature_names_m3).sort_values(ascending=False)

plt.figure(figsize=(12,6))
feat_imp_m3.head(15).plot(kind='barh', color='purple').invert_yaxis()
plt.title('Top 15 Important Features for Model 3 (Customer Impact)')
plt.xlabel('Importance Score')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/3405349506.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Model 4: Customer Complaint Prediction Model (Classification)

In [61]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

In [62]:
master = pd.read_csv('master_dataset.csv')

target_m4 = 'complaint_type'

drop_cols_m4 = [
    'alarm_id',
    'alarm_timestamp',
    'previous_alarm',
    'parent_alarm_id',
    'child_alarm_id',
    'related_alarm_id',
    'event_timestamp',
    'complaint_time',
    'complaint_id',
    target_m4
]

X_m4 = master.drop(columns=drop_cols_m4, errors='ignore')
y_m4 = master[target_m4]

encoder_m4 = LabelEncoder()
y_m4_encoded = encoder_m4.fit_transform(y_m4)
print("Model 4 Target Classes:", list(encoder_m4.classes_))

Model 4 Target Classes: ['5G Not Working', 'Call Drop', 'High Latency', 'No Complaint', 'No Internet', 'Poor Voice', 'SMS Failure', 'Slow Internet']


In [63]:
cat_cols_m4 = X_m4.select_dtypes(include=['object', 'string']).columns
num_cols_m4 = X_m4.select_dtypes(exclude=['object', 'string']).columns

print(f"Categorical Columns ({len(cat_cols_m4)}):", list(cat_cols_m4))
print(f"Numerical Columns ({len(num_cols_m4)}):", list(num_cols_m4))

Categorical Columns (47): ['alarm_type', 'severity', 'technology_x', 'vendor', 'region_x', 'circle_x', 'city_x', 'hub_id', 'router_id', 'bts_id', 'cell_id', 'network_node_x', 'alarm_status', 'root_cause', 'maintenance_required', 'weather_condition', 'power_status', 'next_alarm', 'region_hierarchy', 'circle_hierarchy', 'city_hierarchy', 'hub_id_hierarchy', 'router_id_hierarchy', 'olt_id', 'bts_id_hierarchy', 'cell_id_hierarchy', 'vendor_hierarchy', 'technology_hierarchy', 'weather_event_id', 'region_weather', 'circle_weather', 'city_weather', 'weather_condition_weather', 'weather_severity', 'customer_id', 'network_node_y', 'technology_y', 'customer_priority', 'status', 'region_y', 'circle_y', 'city_y', 'chain_id', 'parent_alarm_type', 'child_alarm_type', 'network_node', 'weekday']
Numerical Columns (24): ['latitude', 'longitude', 'duration_minutes', 'affected_customers', 'resolution_time', 'latitude_hierarchy', 'longitude_hierarchy', 'temperature_c', 'humidity_percent', 'wind_speed_kmph

In [64]:
numeric_transformer_m4 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

categorical_transformer_m4 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_m4 = ColumnTransformer(transformers=[
    ('num', numeric_transformer_m4, num_cols_m4),
    ('cat', categorical_transformer_m4, cat_cols_m4)
])

In [65]:
X_train_m4, X_test_m4, y_train_m4, y_test_m4 = train_test_split(
    X_m4,
    y_m4_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_m4_encoded
)

print('Training features shape:', X_train_m4.shape)
print('Testing features shape:', X_test_m4.shape)

Training features shape: (45166, 71)
Testing features shape: (11292, 71)


In [66]:
pipeline_m4 = Pipeline([
    ('preprocessor', preprocessor_m4),
    ('model', RandomForestClassifier(n_estimators=50, max_depth=20, random_state=42, n_jobs=-1))
])

In [67]:
print('Training Model 4 (Customer Complaint Prediction)...')
pipeline_m4.fit(X_train_m4, y_train_m4)
print('Model 4 trained successfully!')

Training Model 4 (Customer Complaint Prediction)...


Model 4 trained successfully!


In [68]:
y_pred_m4 = pipeline_m4.predict(X_test_m4)

acc_m4 = accuracy_score(y_test_m4, y_pred_m4)
print(f"Model 4 Accuracy: {acc_m4 * 100:.2f}%")
print()

print("Classification Report (Model 4 - Customer Complaint Prediction):")
target_names_m4 = [str(cls) for cls in encoder_m4.classes_]
print(classification_report(y_test_m4, y_pred_m4, target_names=target_names_m4))

Model 4 Accuracy: 62.78%

Classification Report (Model 4 - Customer Complaint Prediction):
                precision    recall  f1-score   support

5G Not Working       0.00      0.00      0.00       604
     Call Drop       1.00      0.00      0.01       600
  High Latency       0.00      0.00      0.00       596
  No Complaint       0.63      1.00      0.77      7081
   No Internet       1.00      0.00      0.00       612
    Poor Voice       1.00      0.01      0.01       586
   SMS Failure       0.00      0.00      0.00       615
 Slow Internet       1.00      0.00      0.00       598

      accuracy                           0.63     11292
     macro avg       0.58      0.13      0.10     11292
  weighted avg       0.61      0.63      0.49     11292



/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [69]:
import matplotlib.pyplot as plt
import seaborn as sns

cm_m4 = confusion_matrix(y_test_m4, y_pred_m4)
plt.figure(figsize=(10, 7))
sns.heatmap(
    cm_m4,
    annot=True,
    fmt='d',
    cmap='Oranges',
    xticklabels=encoder_m4.classes_,
    yticklabels=encoder_m4.classes_
)
plt.xlabel('Predicted Complaint Type')
plt.ylabel('Actual Complaint Type')
plt.title('Confusion Matrix - Model 4 (Customer Complaint Prediction)')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/3363651727.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [70]:
cat_encoder_m4 = pipeline_m4.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
onehot_cols_m4 = list(cat_encoder_m4.get_feature_names_out(cat_cols_m4))
all_feature_names_m4 = list(num_cols_m4) + onehot_cols_m4

importances_m4 = pipeline_m4.named_steps['model'].feature_importances_
feat_imp_m4 = pd.Series(importances_m4, index=all_feature_names_m4).sort_values(ascending=False)

plt.figure(figsize=(12,6))
feat_imp_m4.head(15).plot(kind='barh', color='darkorange').invert_yaxis()
plt.title('Top 15 Important Features for Model 4 (Complaint Prediction)')
plt.xlabel('Importance Score')
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/824643193.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Save Models

In [71]:
import joblib
import os

os.makedirs('models', exist_ok=True)


joblib.dump(pipeline_m1, 'models/root_model.pkl')
joblib.dump(pipeline_m2, 'models/future_alarm.pkl')
joblib.dump(pipeline_m3, 'models/customer_model.pkl')
joblib.dump(pipeline_m4, 'models/complaint_model.pkl')

print("All models saved successfully to the 'models/' directory:")
print("  models/root_model.pkl      -> Model 1: Root Cause Classification")
print("  models/future_alarm.pkl    -> Model 2: Future Alarm Prediction")
print("  models/customer_model.pkl  -> Model 3: Customer Impact Regression")
print("  models/complaint_model.pkl -> Model 4: Customer Complaint Prediction")

All models saved successfully to the 'models/' directory:
  models/root_model.pkl      -> Model 1: Root Cause Classification
  models/future_alarm.pkl    -> Model 2: Future Alarm Prediction
  models/customer_model.pkl  -> Model 3: Customer Impact Regression
  models/complaint_model.pkl -> Model 4: Customer Complaint Prediction


## Feature Importance — All Models

In [72]:
import matplotlib.pyplot as plt
import pandas as pd

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle('Top 15 Feature Importances — All Models', fontsize=16, fontweight='bold', y=1.01)

model_configs = [
    (pipeline_m1, num_cols_m1, cat_cols_m1, 'Model 1: Root Cause Classification',     'steelblue',   axes[0, 0]),
    (pipeline_m2, num_cols_m2, cat_cols_m2, 'Model 2: Future Alarm Prediction',        'seagreen',    axes[0, 1]),
    (pipeline_m3, num_cols_m3, cat_cols_m3, 'Model 3: Customer Impact (Regression)',   'purple',      axes[1, 0]),
    (pipeline_m4, num_cols_m4, cat_cols_m4, 'Model 4: Complaint Prediction',           'darkorange',  axes[1, 1]),
]

for pipeline, num_cols, cat_cols, title, color, ax in model_configs:
    cat_enc = pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
    onehot_cols = list(cat_enc.get_feature_names_out(cat_cols))
    all_feature_names = list(num_cols) + onehot_cols

    importances = pipeline.named_steps["model"].feature_importances_
    feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False).head(15)

    feat_imp[::-1].plot(kind="barh", color=color, ax=ax)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Importance Score")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()

/var/folders/c5/4w9lqh756471fv2jwgbl3vy80000gn/T/ipykernel_30224/4142506099.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
